In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

print("── bronze static ──")
for t in ["gtfs_stops", "gtfs_routes", "gtfs_trips", "gtfs_stop_times",
          "gtfs_calendar", "gtfs_calendar_dates"]:
    df = spark.table(f"transit.bronze.{t}")
    versions = [r[0] for r in df.select("_feed_version").distinct().collect()]
    print(f"  {t:22} {df.count():>12,}   versions: {sorted(versions)}")

print("\n── bronze realtime ──")
rt = spark.table("transit.bronze.rt_vehicle_positions")
s = rt.agg(F.count("*").alias("rows"),
           F.countDistinct("_source_file").alias("files"),
           F.max("_snapshot_ts").alias("latest")).collect()[0]
lag_min = (datetime.now(timezone.utc).timestamp() - s.latest) / 60
print(f"  rt_vehicle_positions   {s.rows:>12,}   files: {s.files:,}   latest: {lag_min:.0f} min ago")

print("\n── ops ──")
spark.table("transit.ops.ingestion_log").orderBy(F.desc("ingested_at")).limit(6).display()

In [0]:
from datetime import datetime, timezone

print("── collectors (landing zone) ──")
stale = []

for feed in ["vehicle_positions", "alerts"]:
    base = f"/Volumes/transit/bronze/landing/rt/{feed}"
    try:
        days = sorted(d.name.rstrip("/") for d in dbutils.fs.ls(base))
    except Exception:
        print(f"  {feed:18} no data yet\n")
        continue
    if not days:
        print(f"  {feed:18} no data yet\n")
        continue

    for d in days[-4:]:
        print(f"  {feed:18} {d}  {len(dbutils.fs.ls(f'{base}/{d}')):>3} files")

    newest = max(f.modificationTime for f in dbutils.fs.ls(f"{base}/{days[-1]}")) / 1000
    mins = (datetime.now(timezone.utc).timestamp() - newest) / 60
    ok = mins < 35
    print(f"  {feed:18} newest file {mins:.0f} min ago  [{'OK' if ok else 'STALE'}]\n")
    if not ok:
        stale.append(feed)

assert not stale, f"collector stale: {stale} - check the job's Runs tab"